# Ajustes NOGSE libres: Hahn, CPMG y contraste

Notebook sencillo para el fantoma de agua de Capiglioni *et al.*. Lee directamente `master.long.parquet`, ajusta solo $D_0$ (con $M_0=1$) y guarda todas las figuras y tablas en `signal_analysis/fittings/`.

Los ajustes usan los modelos NOGSE de `src/models/model_fitting.py`: `M_nogse_free(TE, G, N, x, M0, D0)` para cada señal y la diferencia de dos `M_nogse_free` para el contraste $M_{CPMG}-M_{Hahn}$. Unidades del modelo: `TE` $=T_{NOGSE}$ y `x` en ms, `G` en mT/m y $D_0$ en m²/ms ($1\ \mathrm{mm^2/s}=10^{-9}\ \mathrm{m^2/ms}$). Hahn tiene `x=5 ms`, no `x=0`, por eso el contraste se arma con `M_nogse_free(T, G, N, T/N) - M_nogse_free(T, G, N, x_Hahn)` y no con `NOGSE_contrast_vs_g_free` (que fija `x=0`).

`M_nogse_free` usa lóbulos rectangulares. La sección final compara contra el $b$ exacto de la forma de onda trapezoidal (rampas de 1.2 ms) guardado en `bvalue`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'Data-BIDS').is_dir() and (candidate / 'repos' / 'signal_analysis').is_dir():
            return candidate
    raise FileNotFoundError('No encuentro la raíz del proyecto.')


PROJECT_ROOT = find_project_root()
REPO_ROOT = PROJECT_ROOT / 'repos' / 'signal_analysis'
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from models.model_fitting import _GYRO, M_nogse_free

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
PROJECT_ROOT

## Configuración

Por defecto se usa la rama denoised + Gibbs-removal (`den_gr--manual`). Cambiar solo `ANALYSIS_TAG` permite repetir exactamente el análisis con `raw--manual`. `G` es siempre el gradiente nominal equiespaciado (0, 7, …, 70 mT/m) de la tabla.

In [ ]:
ANALYSIS_TAG = 'den_gr--manual'  # alternativa: 'raw--manual'
ROI = 'water'
STAT = 'avg'
D0_BOUNDS_MM2_S = (1e-4, 1e-2)
MM2_S_TO_M2_MS = 1e-9  # D0 del modelo en m^2/ms

MASTER_PATH = (PROJECT_ROOT / 'Data-BIDS' / 'derivatives' / 'signal_analysis' /
               ANALYSIS_TAG / 'phantoms' / 'nogse_experiments' / 'master.long.parquet')
OUTPUT_ROOT = REPO_ROOT / 'fittings'
SIGNAL_DIR = OUTPUT_ROOT / 'signals'
CONTRAST_DIRECTION_DIR = OUTPUT_ROOT / 'contrasts_by_direction'
CONTRAST_TIME_DIR = OUTPUT_ROOT / 'contrasts_by_time'
TABLE_DIR = OUTPUT_ROOT / 'tables'
for folder in (SIGNAL_DIR, CONTRAST_DIRECTION_DIR, CONTRAST_TIME_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

master = pd.read_parquet(MASTER_PATH)
data = master.loc[(master['stat'].astype(str) == STAT) &
                  (master['roi'].astype(str) == ROI) &
                  (master['type'].astype(str).str.upper().isin(['CPMG', 'HAHN']))].copy()
data['type'] = data['type'].astype(str).str.upper()
data['direction'] = data['direction'].astype(str)
for column in ['td_ms', 'TN', 'N', 'x', 'g', 'bvalue', 'b_step', 'value_norm']:
    data[column] = pd.to_numeric(data[column], errors='coerce')
data = data.dropna(subset=['td_ms', 'TN', 'N', 'x', 'g', 'bvalue', 'b_step', 'value_norm'])
if not np.allclose(data['TN'], data['td_ms']):
    raise ValueError('TN y td_ms no coinciden: revisar la tabla de parámetros.')

print('Master:', MASTER_PATH)
print('Salida:', OUTPUT_ROOT)
display(data.groupby(['type', 'td_ms', 'direction']).size().rename('n').reset_index())

## Modelos y ajuste

`signal_model` y `contrast_model` envuelven `M_nogse_free` con $M_0=1$ (porque `value_norm` vale 1 en $G=0$) y reciben $D_0$ en mm²/s. Se minimiza el error cuadrático en escala de señal normalizada; el ajuste se hace sobre el logaritmo de $D_0$ para mantener $D_0>0$.

In [ ]:
def signal_model(tn_ms, g_mT_m, n_value, x_ms, d0_mm2_s):
    return M_nogse_free(tn_ms, np.asarray(g_mT_m, float), n_value, x_ms, 1.0, d0_mm2_s * MM2_S_TO_M2_MS)


def contrast_model(tn_ms, g_mT_m, n_value, x_cpmg_ms, x_hahn_ms, d0_mm2_s):
    return (signal_model(tn_ms, g_mT_m, n_value, x_cpmg_ms, d0_mm2_s)
            - signal_model(tn_ms, g_mT_m, n_value, x_hahn_ms, d0_mm2_s))


def fit_one_parameter(y, model):
    y = np.asarray(y, float)
    valid = np.isfinite(y)
    lo, hi = np.log(D0_BOUNDS_MM2_S)

    def loss(log_d0):
        prediction = np.asarray(model(np.exp(log_d0)), float)
        return np.mean((y[valid] - prediction[valid]) ** 2)

    result = minimize_scalar(loss, bounds=(lo, hi), method='bounded')
    d0 = float(np.exp(result.x))
    prediction = np.asarray(model(d0), float)
    residual = y[valid] - prediction[valid]
    return {
        'D0_mm2_s': d0,
        'rmse': float(np.sqrt(np.mean(residual**2))),
        'n_points': int(valid.sum()),
        'ok': bool(result.success),
    }


def timing(curve):
    row = curve.iloc[0]
    return float(row['TN']), float(row['N']), float(row['x'])


def d0_label(d0):
    return f'$D_0$={d0 * 1e3:.3f} × 10$^{{-3}}$ mm²/s'


def save_figure(fig, folder, stem):
    fig.savefig(folder / f'{stem}.png', dpi=180, bbox_inches='tight')
    fig.savefig(folder / f'{stem}.pdf', bbox_inches='tight')

## Hahn y CPMG: una figura por medición

Cada figura contiene las tres direcciones y su propio $D_0$ ajustado con `M_nogse_free`. Se generan ocho figuras: Hahn y CPMG para cada uno de los cuatro tiempos.

In [ ]:
direction_colors = {'1': 'tab:blue', '2': 'tab:orange', '3': 'tab:green'}
signal_fit_rows = []

for (sequence_type, td_ms), measurement in data.groupby(['type', 'td_ms'], sort=True):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    for direction, curve in measurement.groupby('direction', sort=True):
        curve = curve.sort_values('g')
        tn_ms, n_value, x_ms = timing(curve)
        g = curve['g'].to_numpy(float)
        y = curve['value_norm'].to_numpy(float)
        fit = fit_one_parameter(y, lambda d0: signal_model(tn_ms, g, n_value, x_ms, d0))
        signal_fit_rows.append({
            'analysis_tag': ANALYSIS_TAG, 'roi': ROI, 'type': sequence_type,
            'td_ms': td_ms, 'direction': direction, 'N': n_value, 'x_ms': x_ms, **fit
        })
        color = direction_colors.get(direction)
        ax.plot(g, y, 'o', color=color, ms=5)
        g_fine = np.linspace(0, g.max(), 400)
        ax.plot(g_fine, signal_model(tn_ms, g_fine, n_value, x_ms, fit['D0_mm2_s']),
                color=color, lw=2, label=f'dir {direction}: {d0_label(fit["D0_mm2_s"])}')
    ax.set(title=f'{sequence_type} — $T_D$={td_ms:g} ms', xlabel='G (mT/m)',
           ylabel='Señal normalizada', xlim=(0, None), ylim=(-0.03, 1.04))
    ax.legend(fontsize=9)
    save_figure(fig, SIGNAL_DIR, f'signal_{sequence_type}_Td-{td_ms:g}ms')
    plt.show()

signal_fits = pd.DataFrame(signal_fit_rows).sort_values(['type', 'td_ms', 'direction'])
signal_fits.to_csv(TABLE_DIR / 'signal_fits.csv', index=False)
display(signal_fits)

## Construcción y ajuste del contraste CPMG − Hahn

Las curvas se emparejan estrictamente por tiempo, dirección y `b_step`. También se verifica que los valores de $G$, $N$ y $T_{NOGSE}$ coincidan. Cada combinación tiempo-dirección tiene un único ajuste de $D_0$, reutilizado en las dos galerías siguientes.

In [ ]:
keys = ['td_ms', 'direction', 'b_step']
paired = ['g', 'bvalue', 'TN', 'N', 'x', 'value_norm']
cpmg = data.loc[data['type'].eq('CPMG'), keys + paired].rename(columns={c: f'{c}_cpmg' for c in paired})
hahn = data.loc[data['type'].eq('HAHN'), keys + paired].rename(columns={c: f'{c}_hahn' for c in paired})
contrast = cpmg.merge(hahn, on=keys, how='inner', validate='one_to_one')
for column in ['g', 'TN', 'N']:
    if not np.allclose(contrast[f'{column}_cpmg'], contrast[f'{column}_hahn'], rtol=1e-6, atol=1e-6):
        raise ValueError(f'CPMG y Hahn no tienen el mismo {column} para todos los puntos emparejados.')
contrast['g'] = contrast['g_cpmg']
contrast['value_contrast'] = contrast['value_norm_cpmg'] - contrast['value_norm_hahn']


def contrast_timing(curve):
    row = curve.iloc[0]
    return float(row['TN_cpmg']), float(row['N_cpmg']), float(row['x_cpmg']), float(row['x_hahn'])


contrast_fit_rows = []
for (td_ms, direction), curve in contrast.groupby(['td_ms', 'direction'], sort=True):
    curve = curve.sort_values('g')
    tn_ms, n_value, x_cpmg, x_hahn = contrast_timing(curve)
    g = curve['g'].to_numpy(float)
    y = curve['value_contrast'].to_numpy(float)
    fit = fit_one_parameter(y, lambda d0: contrast_model(tn_ms, g, n_value, x_cpmg, x_hahn, d0))
    contrast_fit_rows.append({
        'analysis_tag': ANALYSIS_TAG, 'roi': ROI, 'td_ms': td_ms, 'direction': direction,
        'N': n_value, 'x_cpmg_ms': x_cpmg, 'x_hahn_ms': x_hahn, **fit
    })

contrast_fits = pd.DataFrame(contrast_fit_rows).sort_values(['td_ms', 'direction'])
contrast_fits.to_csv(TABLE_DIR / 'contrast_fits.csv', index=False)
display(contrast_fits)

## Contrastes: cuatro tiempos para cada dirección

Cada figura tiene exactamente cuatro curvas, una para cada tiempo disponible.

In [ ]:
time_colors = dict(zip(sorted(contrast['td_ms'].unique()), plt.cm.viridis(np.linspace(0.1, 0.9, contrast['td_ms'].nunique()))))


def plot_contrast_curve(ax, curve, d0, color, label):
    curve = curve.sort_values('g')
    tn_ms, n_value, x_cpmg, x_hahn = contrast_timing(curve)
    ax.plot(curve['g'], curve['value_contrast'], 'o', color=color, ms=5)
    g_fine = np.linspace(0, curve['g'].max(), 400)
    ax.plot(g_fine, contrast_model(tn_ms, g_fine, n_value, x_cpmg, x_hahn, d0), color=color, lw=2,
            label=f'{label}: {d0_label(d0)}')


def contrast_d0(td_ms, direction):
    return contrast_fits.loc[contrast_fits['td_ms'].eq(td_ms) & contrast_fits['direction'].eq(direction), 'D0_mm2_s'].iloc[0]


for direction, panel in contrast.groupby('direction', sort=True):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    for td_ms, curve in panel.groupby('td_ms', sort=True):
        plot_contrast_curve(ax, curve, contrast_d0(td_ms, direction), time_colors[td_ms], f'$T_D$={td_ms:g} ms')
    ax.set(title=f'Contraste CPMG − Hahn — dirección {direction}', xlabel='G (mT/m)',
           ylabel='Contraste normalizado', xlim=(0, None))
    ax.legend(fontsize=8.5)
    save_figure(fig, CONTRAST_DIRECTION_DIR, f'contrast_direction-{direction}')
    plt.show()

## Contrastes: mismo tiempo y distinta dirección

Esta galería muestra directamente cuánto coinciden (o no) las tres direcciones a cada tiempo.

In [ ]:
for td_ms, panel in contrast.groupby('td_ms', sort=True):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    for direction, curve in panel.groupby('direction', sort=True):
        plot_contrast_curve(ax, curve, contrast_d0(td_ms, direction), direction_colors.get(direction), f'dir {direction}')
    ax.set(title=f'Contraste CPMG − Hahn — $T_D$={td_ms:g} ms', xlabel='G (mT/m)',
           ylabel='Contraste normalizado', xlim=(0, None))
    ax.legend(fontsize=9)
    save_figure(fig, CONTRAST_TIME_DIR, f'contrast_Td-{td_ms:g}ms')
    plt.show()

## Diagnóstico de las diferencias entre direcciones

`max_direction_range` es la máxima separación absoluta entre las tres curvas medidas a un mismo $G$. `rms_direction_spread` resume la dispersión respecto de la media direccional. Estos valores se calculan antes del ajuste. `D0_cv_percent` cuantifica la variación de los $D_0$ ajustados entre direcciones.

In [ ]:
qc_rows = []
for (sequence_type, td_ms), panel in data.groupby(['type', 'td_ms'], sort=True):
    wide = panel.pivot(index='b_step', columns='direction', values='value_norm')
    point_ranges = wide.max(axis=1) - wide.min(axis=1)
    point_spread = wide.sub(wide.mean(axis=1), axis=0)
    fitted = signal_fits.loc[(signal_fits['type'].eq(sequence_type)) & signal_fits['td_ms'].eq(td_ms), 'D0_mm2_s']
    qc_rows.append({
        'type': sequence_type, 'td_ms': td_ms,
        'max_direction_range': float(point_ranges.max()),
        'rms_direction_spread': float(np.sqrt(np.nanmean(point_spread.to_numpy()**2))),
        'D0_mean_mm2_s': float(fitted.mean()),
        'D0_cv_percent': float(100 * fitted.std(ddof=1) / fitted.mean()),
    })

direction_qc = pd.DataFrame(qc_rows)
direction_qc.to_csv(TABLE_DIR / 'direction_qc.csv', index=False)
display(direction_qc)

## Comprobación: modelo rectangular vs $b$ exacto de la forma de onda

`M_nogse_free` usa $b_{rect}=\gamma^2 G^2[(N-1)x^3+y^3]/12$, con lóbulos rectangulares. La secuencia juega trapecios con rampas de 1.2 ms dentro de cada período, así que el $b$ real (`bvalue`, traza exacta de la matriz $b$ de la forma de onda) es menor. Esta celda compara tres ajustes de las mismas señales:

- `D0_model`: `M_nogse_free` con el $G$ nominal (los ajustes de arriba).
- `D0_exact_b`: $\exp(-b\,D_0)$ con el `bvalue` exacto (el ajuste anterior del notebook).
- `D0_model_g_eq`: `M_nogse_free` con el gradiente rectangular equivalente $G_{eq}=\sqrt{b/b_{rect}(G=1)}$, que da el mismo $b$ que la forma de onda trapezoidal. Debe coincidir con `D0_exact_b` a precisión numérica: confirma que `M_nogse_free` y el $b$ exacto describen lo mismo y que la diferencia con `D0_model` es solo el factor de rampas `b_exact/b_rect`.

In [ ]:
def b_rect_per_g2(tn_ms, n_value, x_ms):
    # s/mm^2 per (mT/m)^2, the b-value of M_nogse_free (gamma in rad/(ms mT), times in ms)
    y_ms = tn_ms - (n_value - 1) * x_ms
    return _GYRO**2 * ((n_value - 1) * x_ms**3 + y_ms**3) / 12.0 / 1e9


check_rows = []
for (sequence_type, td_ms, direction), curve in data.groupby(['type', 'td_ms', 'direction'], sort=True):
    curve = curve.sort_values('g')
    tn_ms, n_value, x_ms = timing(curve)
    g = curve['g'].to_numpy(float)
    b = curve['bvalue'].to_numpy(float)
    y = curve['value_norm'].to_numpy(float)
    g_eq = np.sqrt(b / b_rect_per_g2(tn_ms, n_value, x_ms))
    model = signal_fits.loc[signal_fits['type'].eq(sequence_type) & signal_fits['td_ms'].eq(td_ms)
                            & signal_fits['direction'].eq(direction), 'D0_mm2_s'].iloc[0]
    exact = fit_one_parameter(y, lambda d0: np.exp(-b * d0))['D0_mm2_s']
    equivalent = fit_one_parameter(y, lambda d0: signal_model(tn_ms, g_eq, n_value, x_ms, d0))['D0_mm2_s']
    positive = g > 0
    check_rows.append({
        'type': sequence_type, 'td_ms': td_ms, 'direction': direction, 'x_ms': x_ms,
        'b_exact_over_b_rect': float(np.median(b[positive] / (b_rect_per_g2(tn_ms, n_value, x_ms) * g[positive]**2))),
        'D0_model': model, 'D0_exact_b': exact, 'D0_model_g_eq': equivalent,
        'model_vs_exact_percent': 100 * (model / exact - 1),
        'g_eq_vs_exact_percent': 100 * (equivalent / exact - 1),
    })

model_check = pd.DataFrame(check_rows)
model_check.to_csv(TABLE_DIR / 'model_vs_exact_b.csv', index=False)
display(model_check)
print('Máxima diferencia D0_model_g_eq vs D0_exact_b (%):', float(model_check['g_eq_vs_exact_percent'].abs().max()))
print('Figuras y tablas guardadas en:', OUTPUT_ROOT)